# 06. 环境变量与 Shell 脚本

AGDR VLA 入门教程 · 第 2 章 · 第 2 周

- **学习目标**：理解 `PATH` 和 `~/.bashrc`，能写带参数、判断和循环的 Shell 脚本
- **前置知识**：[04-管道与重定向](./04-管道与重定向.ipynb)、[05-用户与权限](./05-用户与权限.ipynb)
- **预计学习时长**：1.5 小时

## 变量与环境变量

| 写法 | 含义 |
| --- | --- |
| `name=value` | 定义变量，**等号两边不能有空格** |
| `$name` 或 `${name}` | 使用变量 |
| `export NAME=value` | 变成环境变量，当前 Shell 启动的子程序也能读到 |
| `env` | 列出所有环境变量 |
| `unset NAME` | 删除变量 |

In [ ]:
%%bash
LOCAL_VAR=abc
export EXPORTED_VAR=xyz
echo "当前 Shell: $LOCAL_VAR $EXPORTED_VAR"
bash -c 'echo "子进程: [$LOCAL_VAR] [$EXPORTED_VAR]"'

子进程读不到没有 `export` 的变量。程序读取配置靠的就是环境变量，比如：

```bash
export CUDA_VISIBLE_DEVICES=0         # 让 PyTorch 只看到 0 号 GPU
CUDA_VISIBLE_DEVICES=1 python train.py # 只对这一条命令生效
```

常见的环境变量有 `HOME`（主目录）、`USER`（用户名）、`PATH`（下面马上讲）。

## PATH：Shell 怎么找到命令

输入 `ls` 时，Shell 会依次到 `PATH` 列出的目录里查找，用**第一个**找到的。`PATH` 里的目录用冒号分隔。

In [ ]:
%%bash
echo "$PATH" | tr ':' '\n'
echo ---
which ls

**command not found** 的原因通常有三种：拼写错误、没有安装、安装目录不在 `PATH` 里。

下面把自己的目录加进 `PATH`：

In [ ]:
%%bash
rm -rf /tmp/linux-demo && mkdir -p /tmp/linux-demo/bin && cd /tmp/linux-demo
printf '#!/usr/bin/env bash\necho "我是自己写的命令"\n' > bin/mytool
chmod +x bin/mytool
mytool || echo "--- 还找不到"
export PATH="/tmp/linux-demo/bin:$PATH"
which mytool
mytool

> ⚠️ 修改时一定要带上原来的 `$PATH`。写成 `export PATH=/some/dir` 会丢掉所有系统路径，连 `ls` 都会找不到（重新打开终端即可恢复，除非写进了配置文件）。

把目录放在 `$PATH` **前面**表示优先使用。第 4 章的 Python 环境管理工具，正是靠改 `PATH` 来切换 `python` 的。

## ~/.bashrc：让配置永久生效

`export` 只对当前终端有效，关闭就没了。**`~/.bashrc`** 是每次打开交互式 bash 终端时都会运行的脚本，想永久生效就写到这里。

例如加一个别名和一条 `PATH`（请在终端里操作）：

```bash
nano ~/.bashrc
```

在文件**末尾**添加：

```bash
alias ll='ls -lh'
export PATH="$HOME/.local/bin:$PATH"
```

保存后，修改还不会自动生效，要么重新打开终端，要么运行：

```bash
source ~/.bashrc
```

`source` 表示在**当前 Shell** 里运行这个文件，所以其中的变量和别名会留下来；而 `./脚本` 是在子进程里运行，结束后变量就没了。

> 修改前可以先备份：`cp ~/.bashrc ~/.bashrc.bak`。很多安装程序也会往这里写内容，出现奇怪的问题时，可以来这里排查。

## 第一个 Shell 脚本

脚本就是把命令写进文件，按顺序运行。

In [ ]:
%%bash
cd /tmp/linux-demo
cat > hello.sh <<'EOF'
#!/usr/bin/env bash
# 这是注释
name="AGDR"
echo "你好，$name"
echo "今天是 $(date +%F)"
EOF
chmod +x hello.sh
./hello.sh

- **第一行 `#!/usr/bin/env bash`** 叫 shebang，告诉系统用哪个解释器运行这个文件
- **`chmod +x`** 让它能被直接运行
- **`./hello.sh`** 里的 `./` 不能省略，因为当前目录一般不在 `PATH` 里
- **`$(命令)`** 是命令替换，把命令的输出放到这里，可以赋值给变量

上面的 `cat > 文件 <<'EOF' ... EOF` 只是为了在单元格里创建文件。平时用 nano 或 VS Code 写脚本即可。

## 脚本参数

| 写法 | 含义 |
| --- | --- |
| `$0` | 脚本自身的路径 |
| `$1`、`$2` | 第 1、2 个参数 |
| `$#` | 参数个数 |
| `"$@"` | 所有参数 |

In [ ]:
%%bash
cd /tmp/linux-demo
cat > args.sh <<'EOF'
#!/usr/bin/env bash
echo "脚本: $0"
echo "个数: $#"
echo "第一个: $1"
EOF
chmod +x args.sh
./args.sh "my file.txt" second

**变量要加双引号**。不加的话，`my file.txt` 这样带空格的值会被拆成两个参数。写 `"$1"`，不要写 `$1`。

## 退出码

每个命令结束时都返回一个**退出码**：`0` 表示成功，非 0 表示失败。`$?` 保存上一条命令的退出码。

In [ ]:
%%bash
ls /tmp > /dev/null
echo "成功时: $?"
ls /no/such/dir 2> /dev/null
echo "失败时: $?"

退出码决定了 `&&` 和 `||` 的行为：

| 写法 | 含义 |
| --- | --- |
| `a && b` | `a` 成功才运行 `b` |
| `a \|\| b` | `a` 失败才运行 `b` |
| `exit 2` | 结束脚本，退出码为 2 |
| `echo "错误" >&2` | 把信息输出到 stderr，用于报错 |

## 条件判断

In [ ]:
%%bash
cd /tmp/linux-demo
cat > check.sh <<'EOF'
#!/usr/bin/env bash
if [ "$#" -ne 1 ]; then
    echo "用法: $0 <文件>" >&2
    exit 1
fi

if [ -f "$1" ]; then
    echo "文件存在，共 $(wc -l < "$1") 行"
else
    echo "错误: 文件不存在: $1" >&2
    exit 2
fi
EOF
chmod +x check.sh
./check.sh; echo "退出码: $?"
./check.sh /no/such/file; echo "退出码: $?"
./check.sh check.sh; echo "退出码: $?"

**`[` 后面和 `]` 前面必须有空格**，这是初学者最常犯的错误之一。

| 条件 | 含义 |
| --- | --- |
| `-f 路径` | 是一个文件 |
| `-d 路径` | 是一个目录 |
| `-z "$s"` | 字符串为空 |
| `"$a" = "$b"` | 字符串相等 |
| `"$a" -eq "$b"` | 数字相等 |
| `-ne`、`-gt`、`-lt` | 数字不等、大于、小于 |

## 循环

In [ ]:
%%bash
cd /tmp/linux-demo
touch ep1.log ep2.log ep3.log
for f in ep*.log; do
    echo "处理 $f"
done
for seed in 1 2 3; do
    echo "python train.py --seed $seed"
done

这就是脚本最实用的地方：批量处理文件，或者用不同参数做多次实验。

## 让脚本更安全

脚本默认遇到错误后还会继续运行，下一步可能在错误的状态下执行危险操作。可以在开头加上：

```bash
#!/usr/bin/env bash
set -euo pipefail
```

| 选项 | 作用 |
| --- | --- |
| `-e` | 有命令失败时立即退出 |
| `-u` | 使用未定义的变量时报错 |
| `-o pipefail` | 管道中任何一段失败，都算整条管道失败 |

有个小陷阱：`grep` 没有匹配到内容时退出码是 1，开了 `-e` 后脚本会直接退出。这种情况需要单独处理，比如写成 `grep ... || true`。

## 动手试试

用 nano 或 VS Code 写一个脚本 `/tmp/backup.sh`，不计分：

1. 接受一个目录作为参数；参数个数不对时，向 stderr 输出用法，并以退出码 1 结束
2. 目录不存在时，向 stderr 报错，并以退出码 2 结束
3. 把目录复制成 `目录名.bak`，并输出 `done`

<details>
<summary>参考答案</summary>

```bash
#!/usr/bin/env bash
if [ "$#" -ne 1 ]; then
    echo "用法: $0 <目录>" >&2
    exit 1
fi
if [ ! -d "$1" ]; then
    echo "错误: 目录不存在: $1" >&2
    exit 2
fi
cp -r "$1" "$1.bak"
echo "done"
```

`!` 表示取反。

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 区分普通变量和环境变量 | 变量与环境变量 |
| 解释 `PATH`，并排查 command not found | PATH |
| 用 `~/.bashrc` 和 `source` 保存配置 | ~/.bashrc |
| 写带 shebang、参数、判断、循环的脚本 | Shell 脚本 |
| 用退出码表示成功和失败 | 退出码 |

下一节：[07-进程与系统资源](./07-进程与系统资源.ipynb)